<a href="https://colab.research.google.com/github/dtoralg/TheValley_MDS/blob/main/%5B07%5D%20-%20Ingenieria_de_Variables_I/%5B01%5D%20-%20Notebooks/E3_Pipeline_Completo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E3 · Pipeline completo — Ingenieria de Variables I

## Introduccion

Acabas de hacer varios pasos a mano (limpiar, escalar, codificar). Mañana llegan datos
nuevos: ¿como repites **exactamente** las mismas transformaciones, en el mismo orden y
**sin fugas**? La respuesta es un **Pipeline**.

> Un Pipeline es una **receta** que encadena todos los pasos y los aplica siempre igual.
> El calculo (medias, escalas, categorias) se **aprende solo con train**, y se reaplica a
> test y a produccion. Reproducible, ordenado y facil de auditar.

En este ejercicio montamos numericas + categoricas en un **ColumnTransformer** y lo
enchufamos a una **regresion logistica**, todo dentro de un **Pipeline**. Puntuamos en
datos que apartamos.

## Objetivos del ejercicio

- Entender por que **se imputa/escala despues del split** (y por que el Pipeline lo garantiza).
- Construir pipelines por tipo de variable y combinarlos con **ColumnTransformer**.
- Incluir **target encoding sin fugas** dentro del pipeline (`TargetEncoder`, con validacion interna).
- Entrenar, **evaluar en test** y validar con **cross-validation**.
- Reaplicar el pipeline a **datos nuevos** exactamente igual.

## Descripcion del dataset (fraude con tarjeta)

Trabajamos con un dataset **sintetico y reproducible** de transacciones con tarjeta.
Lo generamos dentro del propio notebook para que sea autocontenido en Colab.
Cada fila es una transaccion con estas variables:

| Variable | Tipo | Descripcion |
|---|---|---|
| `id_cliente` | id | Identificador del cliente |
| `edad` | numerica | Edad del cliente (con algunos huecos) |
| `monto` | numerica | Importe de la transaccion en euros (distribucion sesgada) |
| `pais` | categorica | Pais de la operacion: ES, DE, UK, FR, IT |
| `tipo_tarjeta` | categorica | debito / credito / prepago |
| `comercio` | categorica (ALTA cardinalidad) | Comercio donde se opera (COM_0000 ... COM_0299, con frecuencias muy desiguales) |
| `canal` | categorica | online / presencial (con algunos huecos) |
| `codigo_postal` | pseudo-numerica | Parece numero, pero NO tiene magnitud |
| `fecha` | fecha/hora | Momento de la transaccion |
| `es_fraude` | objetivo (0/1) | 1 si la transaccion fue fraudulenta |

> La probabilidad real de fraude se ha construido en funcion del monto, la hora,
> el canal, el pais y el comercio. Por eso, **las variables que creemos tendran
> senal de verdad** y veremos su efecto en las metricas.

### 1. Importar librerias necesarias

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

### 2. Datos y features deterministas (fila a fila)

In [ ]:
import numpy as np
import pandas as pd

def generar_datos_fraude(n=8000, semilla=42):
    # Dataset sintetico y REPRODUCIBLE de transacciones con tarjeta.
    # La probabilidad de fraude depende de variables reales (monto, hora,
    # canal, pais y comercio): asi la ingenieria de variables tiene senal de verdad.
    rng = np.random.default_rng(semilla)

    # --- Perfil de clientes ---
    n_clientes = 600
    gasto_medio_cliente = rng.lognormal(mean=3.2, sigma=0.5, size=n_clientes)
    edad_por_cliente = rng.integers(18, 80, size=n_clientes)
    id_cliente = rng.integers(0, n_clientes, size=n)

    # --- Comercios (ALTA cardinalidad, con frecuencias MUY desiguales) ---
    n_comercios = 300
    comercios = np.array([f"COM_{i:04d}" for i in range(n_comercios)])
    riesgo_comercio = rng.beta(1.2, 8.0, size=n_comercios)   # casi todos bajos, unos pocos altos
    peso_comercio = 1.0 / np.arange(1, n_comercios + 1)      # ley de potencias: muchos comercios raros
    peso_comercio = peso_comercio / peso_comercio.sum()
    idx_comercio = rng.choice(n_comercios, size=n, p=peso_comercio)

    # --- Categoricas de BAJA cardinalidad ---
    pais = rng.choice(["ES", "DE", "UK", "FR", "IT"], size=n, p=[0.60, 0.12, 0.10, 0.10, 0.08])
    tipo_tarjeta = rng.choice(["debito", "credito", "prepago"], size=n, p=[0.55, 0.40, 0.05])
    canal = rng.choice(["online", "presencial"], size=n, p=[0.45, 0.55])

    # --- Codigo postal (PSEUDO-numerica: parece numero, pero es categorica) ---
    cp_base = rng.choice([28001, 8001, 41001, 46001, 50001], size=n)
    codigo_postal = cp_base + rng.integers(0, 40, size=n)

    # --- Monto (distribucion sesgada con outliers) ---
    monto = rng.lognormal(mean=np.log(gasto_medio_cliente[id_cliente]), sigma=0.8)
    gigantes = rng.random(n) < 0.005                         # unas pocas compras enormes
    monto[gigantes] *= rng.uniform(20, 80, size=int(gigantes.sum()))
    monto = np.round(monto, 2)

    # --- Fecha y hora ---
    inicio = np.datetime64("2024-01-01T00:00")
    minutos = rng.integers(0, 365 * 24 * 60, size=n)
    fecha = pd.to_datetime(inicio + minutos.astype("timedelta64[m]"))
    hora = fecha.hour.to_numpy()
    dia_semana = fecha.dayofweek.to_numpy()

    # --- Probabilidad de fraude: la SENAL vive en estas variables ---
    logit = (
        -4.2
        + 0.45 * (np.log1p(monto) - np.log1p(monto).mean())
        + 1.8 * (hora < 6)
        + 0.7 * (canal == "online")
        + 0.5 * (pais != "ES")
        + 4.0 * riesgo_comercio[idx_comercio]
        + 0.3 * (dia_semana >= 5)
    )
    prob = 1.0 / (1.0 + np.exp(-logit))
    es_fraude = rng.binomial(1, prob)

    df = pd.DataFrame({
        "id_cliente": id_cliente,
        "edad": edad_por_cliente[id_cliente].astype(float),
        "monto": monto,
        "pais": pais,
        "tipo_tarjeta": tipo_tarjeta,
        "comercio": comercios[idx_comercio],
        "canal": canal,
        "codigo_postal": codigo_postal,
        "fecha": fecha,
        "es_fraude": es_fraude,
    })

    # Valores faltantes realistas (para practicar imputacion)
    df.loc[rng.random(n) < 0.05, "edad"] = np.nan
    df.loc[rng.random(n) < 0.03, "canal"] = np.nan
    return df

In [ ]:
df = generar_datos_fraude(n=8000, semilla=42)

# Estas transformaciones son DETERMINISTAS por fila (no aprenden nada del conjunto),
# asi que es seguro calcularlas antes del Pipeline.
df["log_monto"] = np.log1p(df["monto"])
df["hora"] = df["fecha"].dt.hour
df["dia_semana"] = df["fecha"].dt.dayofweek
df["es_madrugada"] = (df["hora"] < 6).astype(int)

df.head()

### 3. Pregunta capciosa: para rellenar huecos con la media, ¿antes o despues del split?

**Despues.** Si calculas la media con train + test, el test se "cuela" en el modelo y el
resultado engaña. Primero separa, aprende SOLO con train y aplica esa media al test.

In [ ]:
X = df.drop(columns=["es_fraude"])
y = df["es_fraude"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)

media_mal = X["edad"].mean()             # MAL: usa train + test
media_bien = X_train["edad"].mean()      # BIEN: solo train
print(f"Media de edad con TODO (mal):  {media_mal:.4f}")
print(f"Media de edad solo TRAIN (ok): {media_bien:.4f}")
print("\nLa diferencia puede parecer pequeña aqui, pero es una FUGA. El Pipeline lo hace bien por nosotros.")

### 4. Definir los grupos de columnas por tipo

In [ ]:
num_cols = ["edad", "monto", "log_monto", "hora", "dia_semana", "es_madrugada"]
cat_baja = ["pais", "tipo_tarjeta", "canal"]       # baja cardinalidad  -> One-Hot
cat_alta = ["comercio"]                             # alta cardinalidad  -> Target encoding
# 'codigo_postal' es pseudo-numerico y 'id_cliente'/'fecha' no entran como tal: los excluimos.
print("Numericas:", num_cols)
print("Categoricas baja card.:", cat_baja)
print("Categoricas alta card.:", cat_alta)

### 5. Un pipeline por tipo de variable + ColumnTransformer

- **Numericas**: imputar (mediana) → escalar.
- **Categoricas baja cardinalidad**: imputar (mas frecuente) → One-Hot.
- **Categoricas alta cardinalidad**: `TargetEncoder` (hace validacion cruzada interna → sin fugas).

In [ ]:
pipe_num = Pipeline([
    ("imputar", SimpleImputer(strategy="median")),
    ("escalar", StandardScaler()),
])

pipe_cat_baja = Pipeline([
    ("imputar", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

pipe_cat_alta = Pipeline([
    ("imputar", SimpleImputer(strategy="most_frequent")),
    ("target", TargetEncoder(target_type="binary", smooth="auto", cv=5)),
])

preprocesado = ColumnTransformer([
    ("num", pipe_num, num_cols),
    ("cat_baja", pipe_cat_baja, cat_baja),
    ("cat_alta", pipe_cat_alta, cat_alta),
])
preprocesado

### 6. Pipeline completo (preprocesado + modelo)

In [ ]:
modelo = Pipeline([
    ("preprocesado", preprocesado),
    ("clasificador", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

modelo.fit(X_train, y_train)
print("Pipeline entrenado correctamente.")

### 7. Evaluar en los datos que apartamos (test)

In [ ]:
proba_test = modelo.predict_proba(X_test)[:, 1]
pred_test = modelo.predict(X_test)

print(f"AUC en test:      {roc_auc_score(y_test, proba_test):.3f}")
print(f"Accuracy en test: {accuracy_score(y_test, pred_test):.3f}")
print()
print(classification_report(y_test, pred_test, target_names=["legitima", "fraude"]))

In [ ]:
cm = confusion_matrix(y_test, pred_test)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["legitima", "fraude"])
disp.plot(cmap=plt.cm.Blues)
plt.title("Matriz de confusion - Pipeline completo")
plt.grid(False)
plt.show()

### 8. Validacion cruzada (el Pipeline evita fugas en cada fold)

Con `cross_val_score`, en cada *fold* el preprocesado se **reaprende solo con el train del fold**.
Por eso la estimacion es honesta.

In [ ]:
scores = cross_val_score(modelo, X_train, y_train, cv=5, scoring="roc_auc")
print("AUC por fold:", np.round(scores, 3))
print(f"AUC media: {scores.mean():.3f} (+/- {scores.std():.3f})")

### 9. Aplicar el pipeline a datos NUEVOS, exactamente igual

Lo que funciona hoy tiene que funcionar mañana con datos nuevos. Generamos un lote nuevo,
creamos las mismas features deterministas y dejamos que el Pipeline aplique el resto.

In [ ]:
df_nuevo = generar_datos_fraude(n=500, semilla=123)
df_nuevo["log_monto"] = np.log1p(df_nuevo["monto"])
df_nuevo["hora"] = df_nuevo["fecha"].dt.hour
df_nuevo["dia_semana"] = df_nuevo["fecha"].dt.dayofweek
df_nuevo["es_madrugada"] = (df_nuevo["hora"] < 6).astype(int)

X_nuevo = df_nuevo.drop(columns=["es_fraude"])
proba_nuevo = modelo.predict_proba(X_nuevo)[:, 1]

resultado = df_nuevo[["monto", "pais", "canal", "comercio"]].copy()
resultado["prob_fraude"] = proba_nuevo.round(3)
print("AUC sobre el lote nuevo:", round(roc_auc_score(df_nuevo["es_fraude"], proba_nuevo), 3))
resultado.sort_values("prob_fraude", ascending=False).head(10)

### Reflexion

1. ¿Por que imputar/escalar **dentro** del Pipeline evita fugas que harias a mano sin darte cuenta?
2. ¿Que ventaja tiene `TargetEncoder` dentro del Pipeline frente a calcular medias a mano?
3. ¿Por que la validacion cruzada con Pipeline da una estimacion mas honesta?
4. Si llega un `pais` nuevo no visto en train, ¿que hace `OneHotEncoder(handle_unknown="ignore")`?
5. ¿Que pasos podrias añadir al Pipeline para mejorar el modelo sin romper la reproducibilidad?